# Fusion: Text + WavLM + Whisper (audios4-CV, prior-adjusted)

Rewritten 2026-04-22. Replaces the earlier LOBO design which averaged across two batches with very different class priors (audios2: 67% cheat, audios4: 18% cheat, audios5: 16% cheat). LOBO-averaged thresholds were pulled toward audios2's high-positive regime — unfit for deployment.

## New training protocol
- **All training data used for feature learning** — audios2 + audios4 combined. audios2's 140 cheating examples are not thrown away.
- **Prior-adjusted class weights** — `scale_pos_weight = (1 - DEPLOY_POS_RATE) / DEPLOY_POS_RATE` (using audios4's ~17% rate) applied everywhere, regardless of the training subset's actual positive rate. The loss function behaves as if the world were 17% positive.
- **Threshold + fusion weights picked on audios4-only 5-fold CV** — audios2 is always on the train side of each fold; the held-out fold is always audios4 rows. This is the deployment-like calibration surface.
- **audios5 evaluated exactly once at the end** with frozen (weights, threshold) from audios4-CV.

## Base models (5, all XGBoost)
| name | features | head |
|---|---|---|
| `wavlm_wp` | frozen WavLM-base-plus whole-audio mean-pool (768d) | XGBoost |
| `whisper_wp_xgb` | frozen Whisper-medium whole-audio mean-pool (1024d) | XGBoost |
| `text_stylo` | 15 stylometric features only | XGBoost |
| `text_top10` | top-10 text features by XGB importance (ranked on audios2-only) | XGBoost |
| `text_top15` | top-15 text features by XGB importance (ranked on audios2-only) | XGBoost |

All heads share `scale_pos_weight = SPW_DEPLOY ~= 4.88`. See `models_explained.md` for what each model does.

## Anti-overfit rules
1. No threshold or α is picked on `audios5`.
2. `audios4-CV` = 5-fold stratified CV on audios4, with audios2 always concatenated onto each fold's training split.
3. Top-10 / top-15 feature rankings are frozen on audios2-only — no audios4 label leakage into feature selection.
4. All fusion search runs on audios4-CV out-of-fold probas.
5. Frozen configs applied to audios5 once. `gap_f1 = cv_f1 - test_f1`. Flag `|gap| > 0.03`.
6. Secondary sanity: cross-batch swap eval (audios2↔4 and 2→5, 4→5).

## Outputs
- `checkpoints_fusion/cv_metrics.csv` — realistic per-fusion numbers on audios4-CV OOF
- `checkpoints_fusion/test_oneshot_metrics.csv` — audios5 at frozen config + CV→test gap
- `checkpoints_fusion/cross_batch.csv` — swap-eval table
- `checkpoints_fusion/frozen_configs.json` — weights + thresholds for every candidate
- `checkpoints_fusion/audios5_full_predictions.csv` + `review_audios5/` — per-file + misclassified copy


In [ ]:
# ================================================================
# CONFIGURATION
# ================================================================
from pathlib import Path

TRAIN_FOLDERS = ["audios2", "audios4"]   # all used for training; audios2 ALWAYS in-train
CV_TARGET     = "audios4"                # 5-fold stratified CV held out here (matches deployment prior)
TEST_FOLDER   = "audios5"                # one-shot evaluation at the very end

# ---- Prior-shift correction ----
# audios2 is 67% cheat, audios4/audios5 ~17%. Training class mix != deployment.
# Fix: use scale_pos_weight that corresponds to the DEPLOYMENT prior, not training.
DEPLOY_POS_RATE = 0.17
SPW_DEPLOY      = (1.0 - DEPLOY_POS_RATE) / DEPLOY_POS_RATE   # ~= 4.88

# Text feature groups considered when ranking top-N features
TEXT_GROUPS = ['stylometric', 'formal_ai', 'disfluency', 'pause']

PREC_TARGETS = [0.80, 0.85, 0.90, 0.95]
RANDOM_SEED  = 42
CV_FOLDS     = 5

NB_DIR   = Path('.').resolve()
SAVE_DIR = NB_DIR / 'checkpoints_fusion'
SAVE_DIR.mkdir(parents=True, exist_ok=True)

LABEL_MAP = {
    'read':1,'cheating':1,'reading':1,'scripted':1,'yes':1,'1':1,1:1,
    'spontaneous':0,'not cheating':0,'not_cheating':0,'no':0,'0':0,0:0,'genuine':0,
}

# Base models: wavlm_wp (XGB), whisper_wp_xgb, text_stylo (XGB), text_top10 (XGB), text_top15 (XGB)
_active = ['wavlm_wp', 'whisper_wp_xgb', 'text_stylo', 'text_top10', 'text_top15']
print(f'Train folders  : {TRAIN_FOLDERS}')
print(f'CV held-out    : {CV_TARGET}  (5-fold stratified; others always in-train)')
print(f'Test folder    : {TEST_FOLDER}')
print(f'Deployment rate: {DEPLOY_POS_RATE:.0%}  ->  SPW_DEPLOY = {SPW_DEPLOY:.2f}')
print(f'Active bases   : {_active}')

In [ ]:
import json, itertools, warnings
import numpy as np
import pandas as pd
import xgboost as xgb
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.isotonic import IsotonicRegression
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import (precision_score, recall_score, f1_score,
                              confusion_matrix)
warnings.filterwarnings('ignore')
np.random.seed(RANDOM_SEED)

GROUPS = {
    'disfluency':  ['filler_rate','filler_count','repetition_rate','repair_rate',
                    'discourse_marker_rate','hedge_rate'],
    'stylometric': ['ttr','mattr','mtld','complex_word_rate','avg_word_length',
                    'n_words','n_unique_words','avg_sentence_length','std_sentence_length',
                    'fragment_rate','n_sentences','self_ref_rate',
                    'noun_rate','verb_rate','adj_rate'],
    'pause':       ['pause_mean','pause_std','pause_median','pause_skew','long_pause_rate',
                    'pause_ratio','n_pauses','pause_regularity',
                    'pause_before_content_ratio','pause_before_function_ratio',
                    'mid_phrase_pause_rate','words_per_sec','articulation_rate',
                    'initial_pause','longest_pause'],
    'formal_ai':   ['formal_transition_count','formal_transition_rate',
                    'ai_phrase_count','ai_phrase_rate'],
}
TEXT_FEATURES = [f for g in TEXT_GROUPS for f in GROUPS[g]]
STYLO_FEATS   = GROUPS['stylometric']
print(f'Candidate text features: {len(TEXT_FEATURES)}')
print(f'Stylometric features   : {len(STYLO_FEATS)}')

## 1. Load cached features (per batch)

One dict `batches[name]` → dataframe with filename, label_int, all features, batch tag. Feature column lists are taken from the first training batch (all batches must share the same schema).

In [ ]:
def load_gt(name):
    gt = pd.read_csv(NB_DIR / f'{name}GT.csv')
    fn_col  = next(c for c in gt.columns if c.lower() in ('filename','file','name'))
    lbl_col = next(c for c in gt.columns if c.lower() in ('label','class','cheating','gt','label_int','ground_truth'))
    gt = gt.rename(columns={fn_col:'filename', lbl_col:'label_raw'})
    gt['label_int'] = gt['label_raw'].map(
        lambda x: LABEL_MAP.get(x, LABEL_MAP.get(str(x).lower().strip(), -1)))
    return gt[gt['label_int'].isin([0,1])][['filename','label_int']]

def _wavlm_wp_path(n):
    p = NB_DIR / f'{n}_whole_pretrained.csv'
    if p.exists(): return p
    alt = NB_DIR / f'{n}_wavlm_whole.csv'
    if alt.exists(): return alt
    raise FileNotFoundError(f'{p} or {alt}')

def load_folder(name):
    gt      = load_gt(name)
    text    = pd.read_csv(NB_DIR / f'{name}_features.csv')
    wavlm   = pd.read_csv(_wavlm_wp_path(name))
    whisper = pd.read_csv(NB_DIR / f'{name}_whisper_whole.csv')
    df = (gt.merge(text,    on='filename', how='inner')
            .merge(wavlm,   on='filename', how='inner', suffixes=('','_wp'))
            .merge(whisper, on='filename', how='inner', suffixes=('','_wh')))
    df['batch'] = name
    return df

batches = {b: load_folder(b) for b in TRAIN_FOLDERS + [TEST_FOLDER]}

_first = batches[TRAIN_FOLDERS[0]]
wp_cols    = [c for c in _first.columns if c.startswith('wavlm_')
              and not c.startswith('wavlm_mean_') and not c.startswith('wavlm_std_')]
wh_cols    = [c for c in _first.columns if c.startswith('whisper_')]
text_cols  = [c for c in TEXT_FEATURES if c in _first.columns]
stylo_cols = [c for c in STYLO_FEATS   if c in _first.columns]

def X_text(df):  return df[text_cols].fillna(0).values
def X_wp(df):    return df[wp_cols].fillna(0).values
def X_wh(df):    return df[wh_cols].fillna(0).values
def X_stylo(df): return df[stylo_cols].fillna(0).values

print(f'Text feats: {len(text_cols)}  |  Stylo: {len(stylo_cols)}  |  WP: {len(wp_cols)}  |  Whisper: {len(wh_cols)}')
for name, df in batches.items():
    y = df['label_int'].values
    print(f'  {name}: n={len(df):4d}  cheat={int((y==1).sum()):3d}  honest={int((y==0).sum()):3d}')

### 1b. Freeze top-10 / top-15 text feature lists

Importance is ranked by an XGBoost trained on the **always-train** batch (audios2) only — this
keeps audios4 labels out of the feature-selection step so the CV numbers below stay honest.
The resulting lists are fixed across all folds of audios4-CV.

In [ ]:
_rank_src = [b for b in TRAIN_FOLDERS if b != CV_TARGET]
_df_rank  = pd.concat([batches[b] for b in _rank_src], ignore_index=True)
_X_rank   = _df_rank[text_cols].fillna(0).values
_y_rank   = _df_rank['label_int'].values

_sc_rank = StandardScaler().fit(_X_rank)
_rkr = xgb.XGBClassifier(
    n_estimators=400, max_depth=4, learning_rate=0.05,
    subsample=0.8, colsample_bytree=0.8, min_child_weight=3,
    scale_pos_weight=float(SPW_DEPLOY), eval_metric='logloss',
    random_state=RANDOM_SEED)
_rkr.fit(_sc_rank.transform(_X_rank), _y_rank)
_imp = pd.Series(_rkr.feature_importances_, index=text_cols).sort_values(ascending=False)

TOP10_FEATS = _imp.head(10).index.tolist()
TOP15_FEATS = _imp.head(15).index.tolist()

def X_top10(df): return df[TOP10_FEATS].fillna(0).values
def X_top15(df): return df[TOP15_FEATS].fillna(0).values

print(f'Ranking source: {_rank_src}  (n={len(_df_rank)})')
print(f'\nTop-10 text features:')
for f in TOP10_FEATS: print(f'  {f:30s}  imp={_imp[f]:.4f}')
print(f'\nTop-15 text features (positions 11-15):')
for f in TOP15_FEATS[10:]: print(f'  {f:30s}  imp={_imp[f]:.4f}')

## 2. Evaluation helpers

`best_f1_on(p, y)` → best-F1 threshold (step 0.01). `metrics_at(p, y, thr)` → prec/rec/f1/confusion. `rec_at_prec(p, y)` → max recall achievable at each precision target.

In [ ]:
def best_f1_on(proba, y, thr_grid=np.arange(0.20, 0.81, 0.01)):
    best_f1, best_thr = -1.0, 0.5
    for thr in thr_grid:
        f = f1_score(y, (proba >= thr).astype(int), zero_division=0)
        if f > best_f1: best_f1, best_thr = f, thr
    return float(best_thr), float(best_f1)

def metrics_at(proba, y, thr):
    pred = (proba >= thr).astype(int)
    cm = confusion_matrix(y, pred, labels=[0,1])
    return dict(
        thr=round(float(thr), 3),
        precision=round(precision_score(y, pred, zero_division=0), 4),
        recall   =round(recall_score   (y, pred, zero_division=0), 4),
        f1       =round(f1_score       (y, pred, zero_division=0), 4),
        tp=int(cm[1,1]), fp=int(cm[0,1]), fn=int(cm[1,0]), tn=int(cm[0,0]),
    )

def rec_at_prec(proba, y, targets=PREC_TARGETS, min_tp=3):
    out = {}
    for t in targets:
        best_rec, best_thr = None, None
        for thr in np.arange(0.99, 0.10, -0.01):
            pred = (proba >= thr).astype(int)
            cm = confusion_matrix(y, pred, labels=[0,1])
            if cm[1,1] < min_tp: continue
            p = precision_score(y, pred, zero_division=0)
            r = recall_score   (y, pred, zero_division=0)
            if p >= t and (best_rec is None or r > best_rec):
                best_rec, best_thr = r, thr
        out[f'rec@P{int(t*100)}'] = round(best_rec, 4) if best_rec is not None else None
        out[f'thr@P{int(t*100)}'] = round(float(best_thr), 3) if best_thr is not None else None
    return out

def evaluate_proba(proba, y, name):
    thr, _ = best_f1_on(proba, y)
    return {'method': name, **metrics_at(proba, y, thr), **rec_at_prec(proba, y)}

## 3. Base-model registry + fit/score utility

`build_registry()` returns `{name: (X_fn, make_clf)}` for the 5 active models (all XGBoost).
Every head uses `scale_pos_weight = SPW_DEPLOY` — the prior-shift fix that makes the loss behave
as if deployment were 17% cheat regardless of the training fold's actual positive rate.

In [ ]:
def make_xgb(n_feats):
    colsample = 0.3 if n_feats > 500 else 0.8
    return xgb.XGBClassifier(
        n_estimators=400, max_depth=4, learning_rate=0.05,
        subsample=0.8, colsample_bytree=colsample, min_child_weight=3,
        scale_pos_weight=float(SPW_DEPLOY), eval_metric='logloss',
        random_state=RANDOM_SEED)

def build_registry():
    # 5 base heads, all XGBoost with SPW_DEPLOY
    return {
        'wavlm_wp':       (X_wp,    lambda: make_xgb(len(wp_cols))),
        'whisper_wp_xgb': (X_wh,    lambda: make_xgb(len(wh_cols))),
        'text_stylo':     (X_stylo, lambda: make_xgb(len(stylo_cols))),
        'text_top10':     (X_top10, lambda: make_xgb(10)),
        'text_top15':     (X_top15, lambda: make_xgb(15)),
    }

def fit_and_score(X_fn, make_clf, df_tr, df_te):
    Xt, yt = X_fn(df_tr), df_tr['label_int'].values
    Xe     = X_fn(df_te)
    sc = StandardScaler().fit(Xt)
    m  = make_clf()
    m.fit(sc.transform(Xt), yt)
    return m.predict_proba(sc.transform(Xe))[:, 1]

print(f'All 5 heads use SPW_DEPLOY = {SPW_DEPLOY:.2f}  (deployment prior = {DEPLOY_POS_RATE:.0%})')
print(f'Heads: {list(build_registry())}')

## 4. audios4-CV predictions (audios2 always in-train)

5-fold stratified CV on `audios4`. For each fold: training split = `audios2` (always) + `audios4[train_idx]`; validation split = `audios4[val_idx]`. Concatenated OOF probas form `cv_scores[model]`, aligned with `cv_y` / `cv_fn`.

These OOF numbers are the deployment-like calibration surface — thresholds and α's picked here are not contaminated by audios2's 67% positive prior.

In [ ]:
always_train_names = [b for b in TRAIN_FOLDERS if b != CV_TARGET]
df_always_tr = pd.concat([batches[b] for b in always_train_names], ignore_index=True) if always_train_names else None
df_cv_target = batches[CV_TARGET].reset_index(drop=True)
y_target     = df_cv_target['label_int'].values

print(f'audios4-CV setup: CV_TARGET={CV_TARGET}  n={len(df_cv_target)}  '
      f'cheat={int((y_target==1).sum())}  honest={int((y_target==0).sum())}')
if df_always_tr is not None:
    y_always = df_always_tr['label_int'].values
    print(f'Always in-train   : {always_train_names}  n={len(df_always_tr)}  '
          f'cheat={int((y_always==1).sum())}  honest={int((y_always==0).sum())}')

_reg_template = build_registry()
cv_scores = {m: np.full(len(df_cv_target), np.nan) for m in _reg_template}

skf = StratifiedKFold(n_splits=CV_FOLDS, shuffle=True, random_state=RANDOM_SEED)
for fold_idx, (tr_idx, va_idx) in enumerate(skf.split(df_cv_target, y_target)):
    df_tr_fold = df_cv_target.iloc[tr_idx]
    df_va_fold = df_cv_target.iloc[va_idx]
    df_tr = pd.concat([df_always_tr, df_tr_fold], ignore_index=True) if df_always_tr is not None else df_tr_fold
    reg = build_registry()
    print(f'  fold {fold_idx+1}/{CV_FOLDS}  n_train={len(df_tr)}  n_val={len(df_va_fold)}  '
          f'val_cheat={int(df_va_fold["label_int"].sum())}')
    for name, (X_fn, make_clf) in reg.items():
        p = fit_and_score(X_fn, make_clf, df_tr, df_va_fold)
        cv_scores[name][va_idx] = p

cv_y  = y_target
cv_fn = df_cv_target['filename'].values
assert not any(np.isnan(v).any() for v in cv_scores.values()), 'CV OOF has NaNs'

print(f'\nCV OOF rows: {len(cv_y)}  cheat={int((cv_y==1).sum())}  honest={int((cv_y==0).sum())}')
print(f'Models in CV: {list(cv_scores)}')

## 5. audios4-CV base-model metrics

Realistic per-model numbers on the deployment-like calibration surface. If `whisper_wp` has much better CV F1 than `wavlm_wp`, expect whisper to dominate fusion; if they're close on CV but whisper wins big on audios5 alone, that win is suspicious.

In [ ]:
cv_base_rows = [evaluate_proba(cv_scores[m], cv_y, f'cv:base:{m}') for m in cv_scores]
cv_base_df = pd.DataFrame(cv_base_rows)
print('audios4-CV base-model metrics:')
print(cv_base_df[['method','thr','precision','recall','f1',
                  'rec@P80','rec@P85','rec@P90','rec@P95']].to_string(index=False))

## 6. Fusion weight search on audios4-CV

For every pair (2-way) and every triple (3-way, 0.1-step simplex grid), sweep weights on CV OOF probas. Pick the (weights, threshold) that maximise **CV F1**. Freeze them. Also fit a stacking `meta_logreg` on CV probas (with nested 5-fold OOF to pick its own best-F1 threshold fairly).

Every entry in `frozen` becomes a candidate to apply to `audios5` in Section 7.

In [ ]:
frozen = []  # each entry: tag, members, weights, thr, cv_f1, cv_prec, cv_rec, cv_rec@P*, proba_cv

def grid_2way(step=0.05):
    return [(round(w, 2), round(1-w, 2)) for w in np.arange(0.0, 1.001, step)]

def grid_3way(step=0.1):
    out = []
    for w1 in np.arange(0, 1.001, step):
        for w2 in np.arange(0, 1.001 - w1 + 1e-9, step):
            w3 = 1.0 - w1 - w2
            if w3 < -1e-9: continue
            out.append((round(w1, 2), round(w2, 2), round(max(0, w3), 2)))
    return out

def record(tag, members, weights, proba_cv):
    thr, _ = best_f1_on(proba_cv, cv_y)
    m = metrics_at(proba_cv, cv_y, thr)
    rp = rec_at_prec(proba_cv, cv_y)
    frozen.append({
        'tag': tag, 'members': list(members), 'weights': list(weights),
        'thr': m['thr'], 'cv_f1': m['f1'],
        'cv_prec': m['precision'], 'cv_rec': m['recall'],
        'cv_rec@P85': rp.get('rec@P85'),
        'cv_rec@P90': rp.get('rec@P90'),
        'cv_rec@P95': rp.get('rec@P95'),
        'proba_cv': proba_cv,
    })

model_names = list(cv_scores)

# ---- 2-way weighted averages ----
for a, b in itertools.combinations(model_names, 2):
    best = None
    for wa, wb in grid_2way(step=0.05):
        p = wa * cv_scores[a] + wb * cv_scores[b]
        thr, f1 = best_f1_on(p, cv_y)
        if best is None or f1 > best['f1']:
            best = {'w': (wa, wb), 'p': p, 'f1': f1}
    record(f'wavg:{a}+{b}', [a, b], best['w'], best['p'])

# ---- 3-way weighted averages ----
if len(model_names) >= 3:
    for trio in itertools.combinations(model_names, 3):
        best = None
        for w in grid_3way(step=0.1):
            p = sum(wi * cv_scores[m] for wi, m in zip(w, trio))
            thr, f1 = best_f1_on(p, cv_y)
            if best is None or f1 > best['f1']:
                best = {'w': w, 'p': p, 'f1': f1}
        record(f'wavg:{"+".join(trio)}', list(trio), best['w'], best['p'])

# ---- Stacking: meta-logreg on CV probas, nested 5-fold OOF for its threshold ----
X_meta_cv = np.column_stack([cv_scores[m] for m in model_names])
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_SEED)
p_meta_oof = np.zeros(len(cv_y))
for tr, va in skf.split(X_meta_cv, cv_y):
    mm = LogisticRegression(C=1.0, class_weight='balanced', max_iter=2000, random_state=RANDOM_SEED)
    mm.fit(X_meta_cv[tr], cv_y[tr])
    p_meta_oof[va] = mm.predict_proba(X_meta_cv[va])[:, 1]

meta_lr = LogisticRegression(C=1.0, class_weight='balanced', max_iter=2000, random_state=RANDOM_SEED)
meta_lr.fit(X_meta_cv, cv_y)

thr_m, f1_m = best_f1_on(p_meta_oof, cv_y)
mm_metrics = metrics_at(p_meta_oof, cv_y, thr_m)
mm_rp      = rec_at_prec(p_meta_oof, cv_y)
frozen.append({
    'tag': 'stack:meta_logreg',
    'members': model_names,
    'weights': [float(w) for w in np.round(meta_lr.coef_[0], 4)],
    'meta_intercept': float(meta_lr.intercept_[0]),
    'thr': mm_metrics['thr'], 'cv_f1': mm_metrics['f1'],
    'cv_prec': mm_metrics['precision'], 'cv_rec': mm_metrics['recall'],
    'cv_rec@P85': mm_rp.get('rec@P85'),
    'cv_rec@P90': mm_rp.get('rec@P90'),
    'cv_rec@P95': mm_rp.get('rec@P95'),
    'proba_cv': p_meta_oof,
})

cv_df = pd.DataFrame([{k: v for k, v in cfg.items() if k != 'proba_cv'} for cfg in frozen])
cv_df = cv_df.sort_values('cv_f1', ascending=False).reset_index(drop=True)
print(f'\nTotal fusion candidates: {len(cv_df)}')
print('\nTop 20 by CV F1:')
cols = ['tag', 'weights', 'thr', 'cv_f1', 'cv_prec', 'cv_rec',
        'cv_rec@P85', 'cv_rec@P90', 'cv_rec@P95']
print(cv_df[cols].head(20).to_string(index=False))

## 6b. Why does rec@P95 differ between runs? (diagnostic)

This cell helps you see WHY the CV number changes when you swap `CV_TARGET`. It prints four things for the current run:

1. **AUCs (ROC + PR)** for every base model and the top fusion on the CV OOF. These are distribution-level separability numbers — they don't care about thresholds. If AUCs are basically the same across runs, the model is *equally good* in both; the rec@P95 gap is just threshold-tail noise, not a real capability difference.
2. **Top-10 highest-scoring negatives** (honest speakers the model thinks are cheaters). These are the "bad apples" that force the 95%-precision threshold up and kill recall. If you see 2–3 negatives scoring near or above your real cheaters, you've found where recall goes to die.
3. **Precision staircase** — walking the threshold from strict to loose, showing how precision falls and recall rises. Lets you see the exact moment where one extra FP drags precision below 95% and the threshold has to snap tighter.
4. **Positive count** — how many true cheaters are even in the CV. With ~15–20 positives, a single missed one costs you 5–7% recall, so small numbers = noisy metric.

Compare this output between the audios4-CV run and the audios5-CV run.

In [ ]:
from sklearn.metrics import roc_auc_score, average_precision_score

print('='*78)
print(f'DIAGNOSTIC — CV_TARGET={CV_TARGET}  n={len(cv_y)}  '
      f'positives={int((cv_y==1).sum())}  negatives={int((cv_y==0).sum())}')
print('='*78)

# ------------------------------------------------------------------
# 1) AUCs for every base model + top fusion
# ------------------------------------------------------------------
print('\n[1] SEPARABILITY (threshold-free) — higher is better')
print(f'{"model":<28}{"ROC-AUC":>10}{"PR-AUC":>10}')
auc_rows = []
for name, p in cv_scores.items():
    roc = roc_auc_score(cv_y, p)
    pr  = average_precision_score(cv_y, p)
    auc_rows.append((name, roc, pr))
    print(f'{name:<28}{roc:>10.4f}{pr:>10.4f}')

top_cfg = max(frozen, key=lambda c: c['cv_f1'])
p_top   = top_cfg['proba_cv']
roc_top = roc_auc_score(cv_y, p_top)
pr_top  = average_precision_score(cv_y, p_top)
print(f'{"FUSION: "+top_cfg["tag"]:<28}{roc_top:>10.4f}{pr_top:>10.4f}')
print(f'  (weights={top_cfg["weights"]}  thr={top_cfg["thr"]})')

# ------------------------------------------------------------------
# 2) Top-10 highest-scoring NEGATIVES (honest speakers the model thinks are cheaters)
# ------------------------------------------------------------------
print('\n[2] TOP-10 HARDEST NEGATIVES on top fusion — these force the 95%-prec threshold up')
print(f'{"rank":>4}  {"score":>7}  {"label":>5}  filename')
order = np.argsort(-p_top)
neg_shown = 0
for rank, idx in enumerate(order, 1):
    if cv_y[idx] == 0:
        neg_shown += 1
        print(f'{neg_shown:>4}  {p_top[idx]:>7.4f}  {"HON":>5}  {cv_fn[idx]}')
        if neg_shown >= 10: break

# For context: what's the lowest-scoring POSITIVE? (real cheaters the model almost missed)
print('\n    For reference — top-5 LOWEST-scoring positives (cheaters the model nearly missed):')
pos_order = np.argsort(p_top)
pos_shown = 0
for idx in pos_order:
    if cv_y[idx] == 1:
        pos_shown += 1
        print(f'{pos_shown:>4}  {p_top[idx]:>7.4f}  {"CHT":>5}  {cv_fn[idx]}')
        if pos_shown >= 5: break

# ------------------------------------------------------------------
# 3) Precision staircase — watch recall at the moment precision drops below 95%
# ------------------------------------------------------------------
print('\n[3] PRECISION STAIRCASE on top fusion (sorted by score, descending)')
print(f'    Shows cumulative TP/FP as threshold loosens. The row where "cum_prec" falls below 0.95')
print(f'    is what determines rec@P95. Look at how many negatives you hit just before it breaks.')
print(f'{"rank":>4}  {"score":>7}  {"label":>5}  {"cum_TP":>7}  {"cum_FP":>7}  {"cum_prec":>9}  {"cum_rec":>8}')
n_pos_total = int((cv_y==1).sum())
cum_tp = cum_fp = 0
broken_95 = False
for rank, idx in enumerate(order, 1):
    if cv_y[idx] == 1: cum_tp += 1
    else:              cum_fp += 1
    cum_prec = cum_tp / (cum_tp + cum_fp)
    cum_rec  = cum_tp / n_pos_total
    mark = ''
    if not broken_95 and cum_prec < 0.95 and (cum_tp + cum_fp) >= 5:
        mark = '   <-- precision drops below 0.95 HERE'
        broken_95 = True
    if rank <= 25 or mark:
        label = 'CHT' if cv_y[idx] == 1 else 'HON'
        print(f'{rank:>4}  {p_top[idx]:>7.4f}  {label:>5}  {cum_tp:>7}  {cum_fp:>7}  {cum_prec:>9.3f}  {cum_rec:>8.3f}{mark}')
    if broken_95 and rank > 25: break

# ------------------------------------------------------------------
# 4) How sensitive is rec@P95 on this CV?
# ------------------------------------------------------------------
print('\n[4] SENSITIVITY — how much does ONE extra false positive cost you at P95?')
print(f'    positives in CV = {n_pos_total}')
print(f'    To hold precision >= 0.95 with K true positives flagged, you can afford at most')
print(f'    floor(K/19) false positives. So every single FP you accept costs up to 19 missed')
print(f'    cheaters before you can take another. Small N makes rec@P95 very jumpy.')
print('='*78)

## 6c. Multi-threshold CV→test transfer (individual models + 2-way + 3-way)

Runs the full pipeline for BOTH configs A (CV=audios4, test=audios5) and B (CV=audios5, test=audios4) and reports, for **every individual base model** and the **top 2-way / top 3-way fusions (ranked separately)**:

For each of 5 CV-chosen thresholds:
- **F1**: threshold that maximises F1 on CV
- **P80 / P85 / P90 / P95**: threshold that gives max recall on CV while precision ≥ target

it shows:
- `thr` — the threshold picked on CV
- `cv_*` — the primary CV metric at that threshold (F1 for F1-strategy; recall for P-strategies)
- `te_*` — the SAME primary metric on test, at the frozen CV threshold
- `teP_*` — test precision at the frozen CV threshold (for P-strategies: did the precision target hold in deployment?)
- `gap_*` — `cv - te` for the primary metric

**Base models (6):**
- `wavlm_wp` — frozen WavLM-base-plus, 768-d whole-audio mean-pool
- `whisper_wp_xgb` — frozen Whisper-medium, 1024-d whole-audio mean-pool
- `text_all` — all 55 text features (8 groups: disfluency, stylometric, pause, suspicious, formal_ai, prosodic, voice_q, perplexity)
- `text_stylo` — 15 stylometric features only
- `text_top10` — top-10 text features (ranked on always-train batches)
- `text_top15` — top-15 text features

A final "gap trend" table shows `gap_F1 → gap_P95` per model so you can see whether the CV→test gap grows, shrinks, or stays flat as you push the precision bar up.

In [ ]:
# ------------------------------------------------------------------
# All 55 text features (same definition as text_cheating_detection.ipynb)
# ------------------------------------------------------------------
ALL_TEXT_FEATURES = [
    # disfluency (6)
    'filler_rate','filler_count','repetition_rate','repair_rate',
    'discourse_marker_rate','hedge_rate',
    # stylometric (15)
    'ttr','mattr','mtld','complex_word_rate','avg_word_length','n_words',
    'n_unique_words','avg_sentence_length','std_sentence_length',
    'fragment_rate','n_sentences','self_ref_rate','noun_rate','verb_rate','adj_rate',
    # pause (15)
    'pause_mean','pause_std','pause_median','pause_skew','long_pause_rate',
    'pause_ratio','n_pauses','pause_regularity','pause_before_content_ratio',
    'pause_before_function_ratio','mid_phrase_pause_rate','words_per_sec',
    'articulation_rate','initial_pause','longest_pause',
    # suspicious (2)
    'suspicious_gap_count','suspicious_gap_ratio',
    # formal_ai (4)
    'formal_transition_count','formal_transition_rate',
    'ai_phrase_count','ai_phrase_rate',
    # prosodic (8)
    'f0_mean','f0_std','f0_range','f0_skew','f0_slope',
    'energy_mean','energy_std','speaking_rate_std',
    # voice_q (3)
    'jitter_local','shimmer_local','hnr_mean',
    # perplexity (2)
    'mean_perplexity','burstiness',
]

STRATEGIES  = ['F1','P80','P85','P90','P95']
PREC_FLOOR  = {'P80':0.80,'P85':0.85,'P90':0.90,'P95':0.95}
TOP_FUSIONS = 10

# ------------------------------------------------------------------
# Threshold pickers
# ------------------------------------------------------------------
def _best_f1_thr(proba, y, grid=np.arange(0.20, 0.81, 0.01)):
    bt, bf = 0.5, -1.0
    for thr in grid:
        f = f1_score(y, (proba >= thr).astype(int), zero_division=0)
        if f > bf: bf, bt = f, float(thr)
    return bt, bf

def _best_rec_at_prec(proba, y, target, min_tp=3):
    best = None
    for thr in np.arange(0.99, 0.10, -0.01):
        pred = (proba >= thr).astype(int)
        cm = confusion_matrix(y, pred, labels=[0,1])
        if cm[1,1] < min_tp: continue
        p = precision_score(y, pred, zero_division=0)
        r = recall_score(y, pred, zero_division=0)
        if p >= target and (best is None or r > best[1]):
            best = (float(thr), float(r), float(p))
    return best if best is not None else (None, None, None)

def _pick_thr(proba, y, strategy):
    if strategy == 'F1':
        thr, f1 = _best_f1_thr(proba, y)
        return thr, f1
    thr, rec, _ = _best_rec_at_prec(proba, y, PREC_FLOOR[strategy])
    return thr, rec

def _metrics_at(proba, y, thr):
    if thr is None: return dict(prec=None, rec=None, f1=None)
    pred = (proba >= thr).astype(int)
    return dict(
        prec=float(precision_score(y, pred, zero_division=0)),
        rec =float(recall_score(y, pred, zero_division=0)),
        f1  =float(f1_score(y, pred, zero_division=0)),
    )

def _fmt(x, p=3):
    if x is None or (isinstance(x, float) and np.isnan(x)): return '   --'
    return f'{x:.{p}f}'

def _build_row(name, p_cv, y_cv, p_te, y_te):
    row = {'model': name}
    for s in STRATEGIES:
        thr, cv_val = _pick_thr(p_cv, y_cv, s)
        te = _metrics_at(p_te, y_te, thr)
        te_val = te['f1'] if s == 'F1' else te['rec']
        gap = (cv_val - te_val) if (cv_val is not None and te_val is not None) else None
        row[f'{s}_thr'] = round(thr, 2) if thr is not None else None
        row[f'{s}_cv']  = round(cv_val, 3) if cv_val is not None else None
        row[f'{s}_te']  = round(te_val, 3) if te_val is not None else None
        if s != 'F1':
            row[f'{s}_teP'] = round(te['prec'], 3) if te['prec'] is not None else None
        row[f'{s}_gap'] = round(gap, 3) if gap is not None else None
    return row

def _cols_for_df():
    cols = ['model']
    for s in STRATEGIES:
        cols += [f'{s}_thr', f'{s}_cv', f'{s}_te']
        if s != 'F1': cols += [f'{s}_teP']
        cols += [f'{s}_gap']
    return cols

# ------------------------------------------------------------------
# Full pipeline (CV + test scoring, 6 base models, 2-way + 3-way fusions)
# ------------------------------------------------------------------
def run_pipeline_ex(train_folders, cv_target, test_folder):
    needed = set(train_folders + [test_folder])
    batches_l = {b: load_folder(b) for b in needed}
    first = batches_l[train_folders[0]]
    wp_cols_l    = [c for c in first.columns if c.startswith('wavlm_')
                    and not c.startswith('wavlm_mean_') and not c.startswith('wavlm_std_')]
    wh_cols_l    = [c for c in first.columns if c.startswith('whisper_')]
    stylo_cols_l = [c for c in STYLO_FEATS       if c in first.columns]
    text_4g_l    = [c for c in TEXT_FEATURES     if c in first.columns]  # 4-group set used for ranking
    text_all_l   = [c for c in ALL_TEXT_FEATURES if c in first.columns]

    rank_src = [b for b in train_folders if b != cv_target]
    df_rank  = pd.concat([batches_l[b] for b in rank_src], ignore_index=True)
    Xr, yr = df_rank[text_4g_l].fillna(0).values, df_rank['label_int'].values
    sc_r = StandardScaler().fit(Xr)
    rkr = xgb.XGBClassifier(n_estimators=400, max_depth=4, learning_rate=0.05,
                             subsample=0.8, colsample_bytree=0.8, min_child_weight=3,
                             scale_pos_weight=float(SPW_DEPLOY), eval_metric='logloss',
                             random_state=RANDOM_SEED)
    rkr.fit(sc_r.transform(Xr), yr)
    imp = pd.Series(rkr.feature_importances_, index=text_4g_l).sort_values(ascending=False)
    top10_l = imp.head(10).index.tolist()
    top15_l = imp.head(15).index.tolist()

    def mk_X(cols): return lambda d: d[cols].fillna(0).values
    registry = {
        'wavlm_wp':       (mk_X(wp_cols_l),    lambda: make_xgb(len(wp_cols_l))),
        'whisper_wp_xgb': (mk_X(wh_cols_l),    lambda: make_xgb(len(wh_cols_l))),
        'text_all':       (mk_X(text_all_l),   lambda: make_xgb(len(text_all_l))),
        'text_stylo':     (mk_X(stylo_cols_l), lambda: make_xgb(len(stylo_cols_l))),
        'text_top10':     (mk_X(top10_l),      lambda: make_xgb(10)),
        'text_top15':     (mk_X(top15_l),      lambda: make_xgb(15)),
    }

    always_tr = [b for b in train_folders if b != cv_target]
    df_always = pd.concat([batches_l[b] for b in always_tr], ignore_index=True) if always_tr else None
    df_cvt = batches_l[cv_target].reset_index(drop=True)
    y_cv_l = df_cvt['label_int'].values

    cv_sc = {m: np.full(len(df_cvt), np.nan) for m in registry}
    skf_l = StratifiedKFold(n_splits=CV_FOLDS, shuffle=True, random_state=RANDOM_SEED)
    for tr_idx, va_idx in skf_l.split(df_cvt, y_cv_l):
        df_tr = pd.concat([df_always, df_cvt.iloc[tr_idx]], ignore_index=True) \
                if df_always is not None else df_cvt.iloc[tr_idx]
        df_va = df_cvt.iloc[va_idx]
        for name, (X_fn, make_clf) in registry.items():
            cv_sc[name][va_idx] = fit_and_score(X_fn, make_clf, df_tr, df_va)

    df_tr_all = pd.concat([batches_l[b] for b in train_folders], ignore_index=True)
    df_te = batches_l[test_folder]
    y_te_l = df_te['label_int'].values
    test_sc = {n: fit_and_score(X_fn, make_clf, df_tr_all, df_te)
               for n, (X_fn, make_clf) in registry.items()}

    # 2-way fusions
    two_way = []
    for a, b in itertools.combinations(list(cv_sc), 2):
        best = None
        for wa in np.arange(0.0, 1.001, 0.05):
            wb = 1.0 - wa
            p = wa*cv_sc[a] + wb*cv_sc[b]
            _, f1 = _best_f1_thr(p, y_cv_l)
            if best is None or f1 > best['f1']:
                best = {'w': (round(float(wa),2), round(float(wb),2)), 'p_cv': p, 'f1': f1}
        p_te = best['w'][0]*test_sc[a] + best['w'][1]*test_sc[b]
        two_way.append({'members': [a,b], 'weights': list(best['w']),
                        'p_cv': best['p_cv'], 'p_te': p_te, 'cv_f1': best['f1']})

    # 3-way fusions
    three_way = []
    model_names = list(cv_sc)
    for trio in itertools.combinations(model_names, 3):
        best = None
        for w1 in np.arange(0, 1.001, 0.1):
            for w2 in np.arange(0, 1.001 - w1 + 1e-9, 0.1):
                w3 = max(0.0, 1.0 - w1 - w2)
                p = w1*cv_sc[trio[0]] + w2*cv_sc[trio[1]] + w3*cv_sc[trio[2]]
                _, f1 = _best_f1_thr(p, y_cv_l)
                if best is None or f1 > best['f1']:
                    best = {'w': (round(float(w1),2), round(float(w2),2), round(float(w3),2)),
                            'p_cv': p, 'f1': f1}
        p_te = sum(w * test_sc[m] for w, m in zip(best['w'], trio))
        three_way.append({'members': list(trio), 'weights': list(best['w']),
                          'p_cv': best['p_cv'], 'p_te': p_te, 'cv_f1': best['f1']})

    return dict(cv_sc=cv_sc, y_cv=y_cv_l, test_sc=test_sc, y_te=y_te_l,
                two_way=two_way, three_way=three_way,
                nfeat_text_all=len(text_all_l), nfeat_text_4g=len(text_4g_l),
                train_folders=train_folders, cv_target=cv_target, test_folder=test_folder)

# ------------------------------------------------------------------
# Report one config
# ------------------------------------------------------------------
def report_config(label, R):
    print('\n' + '='*110)
    print(label)
    print(f'  text_all={R["nfeat_text_all"]} features (all 8 groups)   '
          f'4-group set (stylo/top10/top15 source)={R["nfeat_text_4g"]}')
    print(f'  CV  : n={len(R["y_cv"])}  +{int((R["y_cv"]==1).sum())} / -{int((R["y_cv"]==0).sum())}')
    print(f'  TEST: n={len(R["y_te"])}  +{int((R["y_te"]==1).sum())} / -{int((R["y_te"]==0).sum())}')
    print('='*110)

    cols = _cols_for_df()

    # --- Individual base models ---
    base_rows = [_build_row(m, R['cv_sc'][m], R['y_cv'], R['test_sc'][m], R['y_te'])
                 for m in R['cv_sc']]
    base_df = pd.DataFrame(base_rows)[cols]
    print('\n-- Individual base models (6) --')
    with pd.option_context('display.max_columns', None, 'display.width', 220):
        print(base_df.to_string(index=False, na_rep='  --'))

    # --- 2-way fusions ranked by CV F1 ---
    tw = sorted(R['two_way'], key=lambda x: -x['cv_f1'])[:TOP_FUSIONS]
    tw_rows = []
    for cfg in tw:
        name = '+'.join(cfg['members']) + f' w={cfg["weights"]}'
        tw_rows.append(_build_row(name, cfg['p_cv'], R['y_cv'], cfg['p_te'], R['y_te']))
    tw_df = pd.DataFrame(tw_rows)[cols]
    print(f'\n-- 2-way fusions (top {TOP_FUSIONS} by CV F1) --')
    with pd.option_context('display.max_columns', None, 'display.width', 220):
        print(tw_df.to_string(index=False, na_rep='  --'))

    # --- 3-way fusions ranked by CV F1 ---
    th = sorted(R['three_way'], key=lambda x: -x['cv_f1'])[:TOP_FUSIONS]
    th_rows = []
    for cfg in th:
        name = '+'.join(cfg['members']) + f' w={cfg["weights"]}'
        th_rows.append(_build_row(name, cfg['p_cv'], R['y_cv'], cfg['p_te'], R['y_te']))
    th_df = pd.DataFrame(th_rows)[cols]
    print(f'\n-- 3-way fusions (top {TOP_FUSIONS} by CV F1) --')
    with pd.option_context('display.max_columns', None, 'display.width', 220):
        print(th_df.to_string(index=False, na_rep='  --'))

    # --- Gap trend summary ---
    print('\n-- GAP TREND (cv_metric - test_metric) — does the gap grow as the precision bar rises? --')
    gap_cols = ['model'] + [f'{s}_gap' for s in STRATEGIES]
    merged = pd.concat([base_df[gap_cols].assign(_kind='base'),
                        tw_df[gap_cols].head(3).assign(_kind='2-way'),
                        th_df[gap_cols].head(3).assign(_kind='3-way')],
                       ignore_index=True)
    merged = merged[['_kind','model'] + [f'{s}_gap' for s in STRATEGIES]]
    with pd.option_context('display.max_columns', None, 'display.width', 180):
        print(merged.to_string(index=False, na_rep='  --'))
    print('   Positive gap = overfit to CV (test underperforms).  Negative gap = test overperforms CV.')
    print('   If |gap| grows F1 -> P95, higher precision targets are over-fitting the CV threshold more.')

# ------------------------------------------------------------------
# Run both configs
# ------------------------------------------------------------------
print('Running config A: train=[audios2, audios4]  CV=audios4  test=audios5 ...')
A = run_pipeline_ex(['audios2','audios4'], 'audios4', 'audios5')
print('Running config B: train=[audios2, audios5]  CV=audios5  test=audios4 ...')
B = run_pipeline_ex(['audios2','audios5'], 'audios5', 'audios4')

report_config('CONFIG A — train=[audios2,audios4]  CV=audios4  test=audios5', A)
report_config('CONFIG B — train=[audios2,audios5]  CV=audios5  test=audios4', B)

print('\n' + '='*110)
print('READING GUIDE')
print('='*110)
print('- For each model x strategy, {strategy}_thr is the threshold picked on CV.')
print('- {strategy}_cv = CV metric at that thr (F1 for F1-strategy, recall for P-strategies).')
print('- {strategy}_te = SAME metric on the test batch at the frozen CV threshold.')
print('- {strategy}_teP (P-strategies only) = test precision at the frozen CV threshold.')
print('  If teP < target (e.g. teP<0.95 for P95), the precision target did NOT hold on test.')
print('- {strategy}_gap = cv - te  (positive = CV optimistic; negative = test better than CV).')
print('- Compare gap across F1 / P80 / P85 / P90 / P95 for each model.')
print('  Gap growing with precision = threshold overfits more as you demand higher precision.')


## 7. Error overlap on audios4-CV (Jaccard)

Low Jaccard = complementary models = fusion should work. High Jaccard = redundant → fusion won't add much. Computed on OOF errors at each model's best-F1 threshold.

In [ ]:
errors = {}
for name, p in cv_scores.items():
    thr, _ = best_f1_on(p, cv_y)
    pred = (p >= thr).astype(int)
    errors[name] = set(np.where(pred != cv_y)[0])

names = list(errors.keys())
print('CV error set sizes:', {n: len(errors[n]) for n in names})
print('\nJaccard overlap of CV errors:')
print(f'{"":<12}' + ''.join(f'{n:>14}' for n in names))
for a in names:
    row = [f'{a:<12}']
    for b in names:
        if a == b:
            row.append(f'{1.0:>14.3f}')
        else:
            inter = len(errors[a] & errors[b])
            union = len(errors[a] | errors[b])
            row.append(f'{inter/max(union,1):>14.3f}')
    print(''.join(row))

## 8. Apply frozen configs to TEST (audios5) — one-shot

Fit each base model on the FULL training set (audios2 + audios4), score `audios5`, then apply each frozen (weights, threshold) exactly once. `gap_f1 = cv_f1 - test_f1`.

**Rule:** if `|gap_f1| > 0.03` the config is flagged — either overfit to train (positive gap) or the test batch is materially out-of-distribution (negative gap). Either way, don't blindly deploy.

In [ ]:
df_tr_all = pd.concat([batches[b] for b in TRAIN_FOLDERS], ignore_index=True)
df_te     = batches[TEST_FOLDER]
y_te      = df_te['label_int'].values
fn_te     = df_te['filename'].values
full_reg  = build_registry()  # class weight is SPW_DEPLOY — no dependence on training prior

test_scores = {}
for name, (X_fn, make_clf) in full_reg.items():
    test_scores[name] = fit_and_score(X_fn, make_clf, df_tr_all, df_te)
print(f'Test scored for: {list(test_scores)}')
print(f'Test rows: {len(y_te)}  cheat={int((y_te==1).sum())}  honest={int((y_te==0).sum())}')

# Also keep the per-base test proba for reference evaluation
test_base_rows = [evaluate_proba(test_scores[m], y_te, f'test:base:{m}') for m in test_scores]
test_base_df   = pd.DataFrame(test_base_rows)
print('\nTest base-model metrics (reference, NOT used for threshold selection):')
print(test_base_df[['method','thr','precision','recall','f1',
                    'rec@P80','rec@P85','rec@P90','rec@P95']].to_string(index=False))

# Apply frozen configs to test
rows = []
for cfg in frozen:
    members = cfg['members']; weights = cfg['weights']; thr = cfg['thr']
    if cfg['tag'] == 'stack:meta_logreg':
        X_meta_te = np.column_stack([test_scores[m] for m in members])
        p_te = meta_lr.predict_proba(X_meta_te)[:, 1]
    else:
        p_te = sum(w * test_scores[m] for w, m in zip(weights, members))
    m_te = metrics_at(p_te, y_te, thr)
    rp   = rec_at_prec(p_te, y_te)
    rows.append({
        'tag': cfg['tag'], 'weights': weights, 'thr': thr,
        'cv_f1': cfg['cv_f1'],
        'test_f1': m_te['f1'], 'test_prec': m_te['precision'], 'test_rec': m_te['recall'],
        'gap_f1': round(cfg['cv_f1'] - m_te['f1'], 4),
        'test_rec@P85': rp.get('rec@P85'),
        'test_rec@P90': rp.get('rec@P90'),
        'test_rec@P95': rp.get('rec@P95'),
        'proba_test':  p_te,
    })

one_shot_df = pd.DataFrame(rows).sort_values('cv_f1', ascending=False).reset_index(drop=True)
show_cols = ['tag','weights','thr','cv_f1','test_f1','gap_f1',
             'test_prec','test_rec','test_rec@P85','test_rec@P90','test_rec@P95']
print('\nFrozen configs → test (sorted by CV F1):')
print(one_shot_df[show_cols].head(20).to_string(index=False))

flagged = one_shot_df[one_shot_df['gap_f1'].abs() > 0.03]
print(f'\nFlagged (|gap_f1| > 0.03): {len(flagged)} / {len(one_shot_df)}')
if len(flagged):
    print(flagged[show_cols].to_string(index=False))

## 9. Cross-batch swap evaluation (secondary sanity)

Independent of audios4-CV. Train on ONE batch, test on another. If a fusion wins on `audios2↔audios4` AND `2→5` AND `4→5`, it's robust. If it only wins on `→audios5`, reject.

Now includes every base model in `build_registry()` (whisper + wavlm_sp if enabled) and a best-F1 α sweep per pair.

In [ ]:
def swap_eval(train_name, test_name):
    tr = batches[train_name]; te = batches[test_name]
    ye = te['label_int'].values
    reg = build_registry()  # SPW_DEPLOY-based class weights

    probs = {}
    for name, (X_fn, make_clf) in reg.items():
        probs[name] = fit_and_score(X_fn, make_clf, tr, te)

    rows = []
    for n, p in probs.items():
        thr, _ = best_f1_on(p, ye)
        rows.append({'train': train_name, 'test': test_name,
                     'method': f'base:{n}', **metrics_at(p, ye, thr)})

    # best-F1 α for every pair on this split (sanity view, NOT deployment)
    for a, b in itertools.combinations(probs.keys(), 2):
        best = None
        for wa, wb in grid_2way(step=0.1):
            p = wa * probs[a] + wb * probs[b]
            thr, f1 = best_f1_on(p, ye)
            if best is None or f1 > best['f1']:
                best = {'w': (wa, wb), 'thr': thr, 'f1': f1, 'p': p}
        rows.append({'train': train_name, 'test': test_name,
                     'method': f'wavg:{a}+{b}@a={best["w"][0]}',
                     **metrics_at(best['p'], ye, best['thr'])})
    return rows

swap_rows = []
for a, b in [('audios2','audios4'), ('audios4','audios2'),
             ('audios2','audios5'), ('audios4','audios5')]:
    if a in batches and b in batches:
        swap_rows.extend(swap_eval(a, b))

swap_df = pd.DataFrame(swap_rows)
print('Cross-batch generalisation (best-F1 on each split):')
print(swap_df[['train','test','method','precision','recall','f1']].to_string(index=False))

## 10. Isotonic calibration on the top CV fusion (optional)

Fit isotonic on CV OOF probas (score → true positive rate) and apply to test. After calibration, `score ≥ 0.85` should mean ≈85% precision — useful for picking deployment thresholds by precision target.

In [ ]:
top_tag = cv_df.iloc[0]['tag']
top_cfg = next(cfg for cfg in frozen if cfg['tag'] == top_tag)
print(f'Calibrating top CV fusion: {top_tag}')

if top_tag == 'stack:meta_logreg':
    p_cv = top_cfg['proba_cv']
    p_te = meta_lr.predict_proba(np.column_stack([test_scores[m] for m in top_cfg['members']]))[:, 1]
else:
    p_cv = sum(w * cv_scores[m] for w, m in zip(top_cfg['weights'], top_cfg['members']))
    p_te = sum(w * test_scores[m] for w, m in zip(top_cfg['weights'], top_cfg['members']))

iso = IsotonicRegression(out_of_bounds='clip').fit(p_cv, cv_y)
p_cv_c = iso.transform(p_cv)
p_te_c = iso.transform(p_te)

print('\nReliability check — apply calibrated-score threshold = target precision, measure ACTUAL precision:')
print(f'{"thr":>6}  {"test_prec":>10}  {"test_rec":>9}  {"tp":>4}  {"fp":>4}  {"fn":>4}')
for t in [0.50, 0.60, 0.70, 0.80, 0.85, 0.90, 0.95]:
    pred = (p_te_c >= t).astype(int)
    cm = confusion_matrix(y_te, pred, labels=[0,1])
    if cm[1,1] < 3:
        continue
    prec = precision_score(y_te, pred, zero_division=0)
    rec  = recall_score(y_te, pred, zero_division=0)
    print(f'{t:>6.2f}  {prec:>10.4f}  {rec:>9.4f}  {cm[1,1]:>4}  {cm[0,1]:>4}  {cm[1,0]:>4}')

## 11. Save frozen configs + misclassification review

Persists CV table, one-shot test table, cross-batch table, and the per-file predictions for the top CV fusion. Copies misclassified audio files under `review_audios5/{FP,FN,CORRECT_BORDERLINE}/` for manual review.

In [ ]:
import shutil

# Save tables
cv_df.to_csv(SAVE_DIR / 'cv_metrics.csv', index=False)
one_shot_df.drop(columns=['proba_test']).to_csv(SAVE_DIR / 'test_oneshot_metrics.csv', index=False)
swap_df.to_csv(SAVE_DIR / 'cross_batch.csv', index=False)

# Save frozen configs (drop proba arrays for clean json)
serial_frozen = []
for cfg in frozen:
    d = {k: v for k, v in cfg.items() if k != 'proba_cv'}
    serial_frozen.append(d)
with open(SAVE_DIR / 'frozen_configs.json', 'w') as f:
    json.dump(serial_frozen, f, indent=2, default=str)

# Per-file review for the top CV fusion
top_row = one_shot_df.iloc[0]
proba_test = top_row['proba_test']; thr = top_row['thr']

review = pd.DataFrame({
    'filename': fn_te,
    'current_gt': y_te.astype(int),
    'score':     np.round(proba_test, 4),
    'score_cal': np.round(p_te_c, 4),
    'pred':      (proba_test >= thr).astype(int),
})
for m, p in test_scores.items():
    review[f'p_{m}'] = np.round(p, 4)

def err_type(row):
    if row['current_gt'] == row['pred']:
        if abs(row['score'] - thr) <= 0.08: return 'CORRECT_BORDERLINE'
        return 'CORRECT'
    return 'FP' if row['current_gt'] == 0 else 'FN'

review['error_type'] = review.apply(err_type, axis=1)
review = review.sort_values('score', ascending=False).reset_index(drop=True)
review['rank'] = review.index + 1
review['correct_gt'] = ''
review['notes'] = ''

col_order = ['rank', 'filename', 'current_gt', 'score', 'score_cal', 'pred', 'error_type'] \
            + [f'p_{m}' for m in test_scores] \
            + ['correct_gt', 'notes']
review = review[col_order]

full_pred_path = SAVE_DIR / 'audios5_full_predictions.csv'
review.to_csv(full_pred_path, index=False)
print(f'Full predictions -> {full_pred_path}')

# Misclassified copy
REVIEW_DIR = NB_DIR / 'review_audios5'
REVIEW_DIR.mkdir(parents=True, exist_ok=True)
misc = review[review['error_type'].isin(['FP','FN','CORRECT_BORDERLINE'])].copy()
misc = misc.sort_values(['error_type','score'], ascending=[True, False]).reset_index(drop=True)
misc.to_csv(REVIEW_DIR / 'audios5_misclassified.csv', index=False)

src_candidates = [NB_DIR / TEST_FOLDER, NB_DIR.parent / TEST_FOLDER]
src_dir = next((p for p in src_candidates if p.exists()), None)
copied, missing = 0, []
if src_dir:
    for _, row in misc.iterrows():
        fn = row['filename']
        sub = REVIEW_DIR / row['error_type']; sub.mkdir(parents=True, exist_ok=True)
        src_file = src_dir / fn
        if not src_file.exists():
            for ext in ('.wav','.mp3','.m4a','.flac','.ogg'):
                alt = src_dir / (fn + ext)
                if alt.exists(): src_file = alt; break
        if src_file.exists():
            dest = sub / f"r{int(row['rank']):03d}_s{row['score']:.3f}_gt{int(row['current_gt'])}_{src_file.name}"
            shutil.copy2(src_file, dest); copied += 1
        else:
            missing.append(fn)
    print(f'Copied {copied} misclassified files to {REVIEW_DIR} (missing: {len(missing)})')
else:
    print(f'[WARN] {TEST_FOLDER}/ not found — copy files manually.')

print('\n' + '='*70)
print('Summary — top CV fusion')
print('='*70)
print(f'  tag     : {top_row["tag"]}')
print(f'  weights : {top_row["weights"]}')
print(f'  thr     : {top_row["thr"]}')
print(f'  CV F1   : {top_row["cv_f1"]:.4f}')
print(f'  Test F1 : {top_row["test_f1"]:.4f}   (gap: {top_row["gap_f1"]:+.4f})')
print(f'  Test P/R: {top_row["test_prec"]:.4f} / {top_row["test_rec"]:.4f}')
print(f'  rec@P85={top_row["test_rec@P85"]}  rec@P90={top_row["test_rec@P90"]}  rec@P95={top_row["test_rec@P95"]}')
print(f'\nArtifacts in: {SAVE_DIR}')